# CA-PFG — Regenerate manuscript Figures 2 and 3 from held-out TEST metrics

**Goal:** Produce the two figures in the revised IJITCS manuscript from the **test-only results** of GAPFA12 v3. This is a plotting notebook, **not an additional experiment**. It needs neither GPU, image datasets, feature caches, nor model re-training.

- **Fig. 2:** Mean test Macro-F1 vs. mean test ECE under DINOv2-Small and ResNet-50, four method variants, with arrows showing the sequence of components.
- **Fig. 3:** Three descriptive panels (mean test Macro-F1, NLL, ECE), with four methods and both backbones.
- **Input (preferred):** `normalized_all_results_TEST_ONLY.csv` (480 rows) from the corrected Notebook 12 v3.
- **Alternate input:** `GAPFA12_v3_TEST_Corrected_Full_Results.zip`; its internal `normalized_all_results.csv` is accepted **only** because explicit `test_*` columns are present.
- **Safeguard:** All calculations use `test_macro_f1`, `test_nll`, and `test_ece` exclusively; the notebook never uses validation metrics or the ambiguous unprefixed columns.
- **Outputs:** PNG (450 dpi), vector PDF and SVG, grouped means CSV, source audit JSON, ZIP.

**Expected design:** 2 backbones × 2 datasets × 3 shot levels × 10 seeds × 4 methods = **480 records**. Each backbone-method group must contain **60 matched tasks**. All four methods must have identical task keys. No missing/non-finite test metrics allowed.


In [ ]:
from pathlib import Path
from io import BytesIO
import hashlib
import json
import os
import zipfile

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# ---------- INPUT: ordinarily no edits needed ----------
# Optional: supply an exact full path to the corrected CSV if automatic discovery
# finds more than one copy, e.g. '/kaggle/input/my-dataset/normalized_all_results_TEST_ONLY.csv'.
CSV_OVERRIDE = ""

# Optional alternative input: an exact ZIP path. Leave blank for auto-discovery.
ZIP_OVERRIDE = ""

# ---------- OUTPUT ----------
WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
OUT = WORK / 'CAPFG_Figures_2_3_TEST_Corrected'
OUT.mkdir(parents=True, exist_ok=True)

DPI = 450  # journal requires >=300 dpi
BACKBONES = ['DINOv2-Small', 'ResNet-50']
METHODS = ['LR-Real', 'PFG-UnweightedLR', 'PFG-NW-F1Only', 'CA-PFG-MO']
SHORT = {
    'LR-Real': 'LR-Real',
    'PFG-UnweightedLR': 'PFG-Unweighted',
    'PFG-NW-F1Only': 'PFG-NW-F1',
    'CA-PFG-MO': 'CA-PFG-MO',
}
METRICS = {'macro_f1': 'Macro-F1', 'nll': 'NLL', 'ece': 'ECE'}

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 11.5,
    'axes.labelsize': 10.5,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
    'savefig.dpi': DPI,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('Output directory:', OUT)
print('Matplotlib:', matplotlib.__version__)


In [ ]:
def _find_input_path(exact_name: str) -> Path | None:
    roots = [Path('/kaggle/input'), Path('/mnt/data')]
    found = []
    for root in roots:
        if root.exists():
            found.extend(p for p in root.rglob(exact_name) if p.is_file())
    found = sorted(set(found), key=lambda p: str(p))
    if len(found) > 1:
        raise RuntimeError(
            f'Multiple files called {exact_name} were found. Set CSV_OVERRIDE or ZIP_OVERRIDE explicitly:\n' +
            '\n'.join(map(str, found))
        )
    return found[0] if found else None


def _load_source():
    if CSV_OVERRIDE:
        p = Path(CSV_OVERRIDE)
        if not p.is_file():
            raise FileNotFoundError(f'CSV_OVERRIDE path does not exist: {p}')
        b = p.read_bytes()
        return b, str(p), 'csv'

    if ZIP_OVERRIDE:
        zpath = Path(ZIP_OVERRIDE)
        if not zpath.is_file():
            raise FileNotFoundError(f'ZIP_OVERRIDE path does not exist: {zpath}')
    else:
        p = _find_input_path('normalized_all_results_TEST_ONLY.csv')
        if p is not None:
            b = p.read_bytes()
            return b, str(p), 'csv'
        zpath = _find_input_path('GAPFA12_v3_TEST_Corrected_Full_Results.zip')
        if zpath is None:
            raise FileNotFoundError(
                'Cannot find normalized_all_results_TEST_ONLY.csv or the corrected GAPFA12 v3 ZIP. '
                'Attach one of them through Kaggle Add Input, or set CSV_OVERRIDE / ZIP_OVERRIDE.'
            )

    with zipfile.ZipFile(zpath) as z:
        matching = [name for name in z.namelist()
                    if Path(name).name in {'normalized_all_results_TEST_ONLY.csv', 'normalized_all_results.csv'}]
        if not matching:
            raise RuntimeError(f'No per-task results CSV inside {zpath}')
        preferred = sorted(matching, key=lambda p: (
            0 if Path(p).name == 'normalized_all_results_TEST_ONLY.csv' else 1,
            len(Path(p).parts), p
        ))[0]
        b = z.read(preferred)
    return b, f'{zpath}::{preferred}', 'zip'


csv_bytes, SOURCE_NAME, SOURCE_KIND = _load_source()
raw = pd.read_csv(BytesIO(csv_bytes))

required = {'backbone', 'dataset', 'shot', 'seed', 'method',
            'test_macro_f1', 'test_nll', 'test_ece'}
missing = required.difference(raw.columns)
if missing:
    raise AssertionError(
        'Required explicit TEST columns missing: ' + str(sorted(missing)) +
        '. Do not substitute validation_* or unprefixed columns.'
    )

# Explicitly isolate held-out TEST columns; do not use raw["macro_f1"], etc.
df = raw[['backbone', 'dataset', 'shot', 'seed', 'method',
          'test_macro_f1', 'test_nll', 'test_ece']].copy()
df = df.rename(columns={
    'test_macro_f1': 'macro_f1',
    'test_nll': 'nll',
    'test_ece': 'ece'
})

for col in ('macro_f1', 'nll', 'ece'):
    df[col] = pd.to_numeric(df[col], errors='raise')
    if not np.isfinite(df[col].to_numpy()).all():
        raise AssertionError(f'Missing/nonfinite TEST metric {col}')

if df.shape[0] != 480:
    raise AssertionError(f'Expected 480 task-method records, got {len(df)}')
if set(df['backbone']) != set(BACKBONES):
    raise AssertionError(f'Unexpected backbones: {sorted(df.backbone.unique())}')
if set(df['method']) != set(METHODS):
    raise AssertionError(f'Unexpected methods: {sorted(df.method.unique())}')
if set(df['dataset']) != {'plantdoc', 'plantvillage'}:
    raise AssertionError(f'Unexpected datasets: {sorted(df.dataset.unique())}')
if set(df['shot']) != {5, 10, 20}:
    raise AssertionError(f'Unexpected shot settings: {sorted(df.shot.unique())}')
if not df['macro_f1'].between(0, 1).all() or not df['ece'].between(0, 1).all() or (df['nll'] < 0).any():
    raise AssertionError('Some test metrics lie outside their valid ranges')
keys = ['backbone', 'dataset', 'shot', 'seed', 'method']
if df.duplicated(keys).any():
    raise AssertionError('Duplicate backbone/dataset/shot/seed/method keys')

for backbone in BACKBONES:
    ref_keys = None
    for method in METHODS:
        g = df[(df['backbone'] == backbone) & (df['method'] == method)]
        if len(g) != 60:
            raise AssertionError(f'Expected 60 tasks for {backbone}/{method}, got {len(g)}')
        task_keys = set(map(tuple, g[['dataset', 'shot', 'seed']].to_numpy()))
        if ref_keys is None:
            ref_keys = task_keys
        elif task_keys != ref_keys:
            raise AssertionError(f'Task mismatch for {backbone}/{method}')

source_sha256 = hashlib.sha256(csv_bytes).hexdigest()
print('[PASS] Test-only source:', SOURCE_NAME)
print('[PASS] SHA256:', source_sha256)
print('[PASS] 480 rows; 2 backbones; 4 methods; 60 matched tasks per method/backbone')


In [ ]:
MEANS = (df.groupby(['backbone', 'method'], as_index=False)
           .agg(n_tasks=('macro_f1', 'size'),
                macro_f1_mean=('macro_f1', 'mean'),
                macro_f1_sd=('macro_f1', 'std'),
                nll_mean=('nll', 'mean'),
                nll_sd=('nll', 'std'),
                ece_mean=('ece', 'mean'),
                ece_sd=('ece', 'std')))
MEANS['backbone'] = pd.Categorical(MEANS['backbone'], categories=BACKBONES, ordered=True)
MEANS['method'] = pd.Categorical(MEANS['method'], categories=METHODS, ordered=True)
MEANS = MEANS.sort_values(['backbone', 'method']).reset_index(drop=True)
MEANS.to_csv(OUT / 'figure_2_3_source_TEST_means.csv', index=False)

# Basic cross-check against independently calculated expected held-out test means.
expected_capfg = {
    'DINOv2-Small': {'macro_f1_mean': 0.664251, 'nll_mean': 1.038570, 'ece_mean': 0.084763},
    'ResNet-50': {'macro_f1_mean': 0.5443, 'nll_mean': 1.5255, 'ece_mean': 0.0975},
}
for backbone, expected in expected_capfg.items():
    g = MEANS[(MEANS['backbone'] == backbone) & (MEANS['method'] == 'CA-PFG-MO')].iloc[0]
    for k, val in expected.items():
        if not np.isclose(float(g[k]), val, atol=7e-5, rtol=0):
            raise AssertionError(
                f'Unexpected {backbone} / {k}: {float(g[k]):.6f} (expected approx. {val:.6f}). '
                'Are you using the final TEST-corrected output?'
            )

with pd.option_context('display.float_format', '{:.5f}'.format):
    print(MEANS[['backbone', 'method', 'n_tasks', 'macro_f1_mean', 'nll_mean', 'ece_mean']].to_string(index=False))
print('[PASS] Means verified against the corrected held-out TEST summary')


In [ ]:
# Fig. 2 — descriptive test Macro-F1 vs. ECE operating points
fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.5), constrained_layout=True)
markers = ['o', 's', 'D', 'P']

# Label offsets are expressed in display points, not data units.
label_offsets = {
    'DINOv2-Small': [(-52, -11), (8, 11), (-77, 6), (-53, -12)],
    'ResNet-50': [(-55, 11), (8, 10), (-69, -13), (7, -12)],
}

for ax, backbone in zip(axes, BACKBONES):
    g = MEANS[MEANS['backbone'] == backbone].set_index('method').loc[METHODS]
    x = g['macro_f1_mean'].to_numpy(dtype=float)
    y = g['ece_mean'].to_numpy(dtype=float)

    # Neutral connecting arrows show methodological order, NOT monotonic gains.
    for j in range(len(METHODS) - 1):
        ax.annotate('', xy=(x[j+1], y[j+1]), xytext=(x[j], y[j]),
                    arrowprops=dict(arrowstyle='->', lw=1.35,
                                    mutation_scale=11, shrinkA=7, shrinkB=7, alpha=0.68),
                    zorder=1)
    for i, method in enumerate(METHODS):
        ax.scatter(x[i], y[i], s=68, marker=markers[i], zorder=3)
        ax.annotate(SHORT[method], (x[i], y[i]), xytext=label_offsets[backbone][i],
                    textcoords='offset points', fontsize=8.6, zorder=4)

    xr = float(max(x) - min(x)) or 0.001
    yr = float(max(y) - min(y)) or 0.01
    ax.set_xlim(float(min(x)) - 0.17*xr, float(max(x)) + 0.22*xr)
    ax.set_ylim(float(min(y)) - 0.21*yr, float(max(y)) + 0.20*yr)
    ax.set_title(backbone, fontweight='semibold')
    ax.set_xlabel('Mean test Macro-F1 (higher is better)')
    ax.set_ylabel('Mean test ECE (lower is better)')
    ax.grid(alpha=0.20, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

# Caption should be placed below the figure in Word, not baked into the image.
base = OUT / 'Fig2_TEST_MacroF1_ECE'
for extension in ('png', 'pdf', 'svg'):
    fig.savefig(base.with_suffix('.' + extension), dpi=DPI,
                bbox_inches='tight', pad_inches=0.15)
plt.show()
plt.close(fig)
print('Saved Fig. 2:', base)


In [ ]:
# Fig. 3 — three descriptive test-metric panels, both backbones, 4 methods
fig, axes = plt.subplots(1, 3, figsize=(12.0, 4.6), constrained_layout=True)
metrics = [
    ('macro_f1_mean', 'Macro-F1', 'Higher is better'),
    ('nll_mean', 'NLL', 'Lower is better'),
    ('ece_mean', 'ECE', 'Lower is better'),
]
x = np.arange(len(METHODS))

for ax, (metric, ylabel, orient) in zip(axes, metrics):
    for backbone, marker, line in zip(BACKBONES, ['o', 's'], ['-', '--']):
        g = MEANS[MEANS['backbone'] == backbone].set_index('method').loc[METHODS]
        y = g[metric].to_numpy(dtype=float)
        ax.plot(x, y, linestyle=line, marker=marker, markersize=6.4,
                linewidth=1.85, label=backbone)
    ax.set_xticks(x, [SHORT[m] for m in METHODS], rotation=29, ha='right')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} — {orient}', fontweight='semibold')
    ax.grid(axis='y', alpha=0.20, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    # Descriptive line plots use a zoomed y-axis to expose small method differences.
    yl, yh = ax.get_ylim()
    if yl < 0:
        ax.set_ylim(0, yh)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='outside upper center', ncol=2, frameon=False)

base = OUT / 'Fig3_TEST_Overall_Method_Comparison'
for extension in ('png', 'pdf', 'svg'):
    fig.savefig(base.with_suffix('.' + extension), dpi=DPI,
                bbox_inches='tight', pad_inches=0.15)
plt.show()
plt.close(fig)
print('Saved Fig. 3:', base)


In [ ]:
audit = {
    'input_source': SOURCE_NAME,
    'input_csv_sha256': source_sha256,
    'source_kind': SOURCE_KIND,
    'source_columns_used': ['test_macro_f1', 'test_nll', 'test_ece'],
    'validation_columns_used': [],
    'rows': int(len(df)),
    'backbones': BACKBONES,
    'methods': METHODS,
    'matched_tasks_per_backbone_method': 60,
    'png_dpi': DPI,
    'notes': [
        'Descriptive visualizations only; do not interpret differences as statistical significance.',
        'Fig. 2 arrows show method order, not monotonic improvement.',
        'Fig. 3 uses non-zero / zoomed y-axes where appropriate; inspect numerical tables for effect sizes.',
    ],
}
(OUT / 'figure_data_provenance.json').write_text(
    json.dumps(audit, indent=2, ensure_ascii=False), encoding='utf-8'
)

captions = """Fig. 2. Test Macro-F1–ECE trade-off for four method variants on DINOv2-Small and ResNet-50. Each point is the mean over 60 matched held-out test tasks per backbone. Higher Macro-F1 and lower ECE are preferable; arrows show the methodological sequence, not a monotonic performance guarantee.

Fig. 3. Held-out test comparison of four method variants under frozen DINOv2-Small and ResNet-50 representations. Panels show mean Macro-F1, NLL, and ECE over 60 matched tasks per backbone; paired inferential results are presented separately in Table 4.
"""
(OUT / 'figure_captions_for_manuscript.txt').write_text(captions, encoding='utf-8')

required_outputs = [
    'Fig2_TEST_MacroF1_ECE.png', 'Fig2_TEST_MacroF1_ECE.pdf', 'Fig2_TEST_MacroF1_ECE.svg',
    'Fig3_TEST_Overall_Method_Comparison.png', 'Fig3_TEST_Overall_Method_Comparison.pdf',
    'Fig3_TEST_Overall_Method_Comparison.svg',
    'figure_2_3_source_TEST_means.csv', 'figure_data_provenance.json',
    'figure_captions_for_manuscript.txt',
]
for name in required_outputs:
    assert (OUT/name).is_file(), f'Missing output: {name}'

zip_path = WORK / 'CAPFG_Figures_2_3_TEST_Corrected.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for name in required_outputs:
        zf.write(OUT / name, arcname=name)
print('[COMPLETE] 2 figures × PNG/PDF/SVG + mean table, provenance and captions')
print('ZIP:', zip_path, 'size:', round(zip_path.stat().st_size / 1024, 1), 'KiB')
print('Files:')
for name in required_outputs:
    print(' -', name)


### Suggested usage in the revision

- Replace only **Fig. 2** and **Fig. 3** in the Word manuscript. **Algorithm 1** is not a figure.
- Choose **PNG 450 dpi** or **SVG** for Word. PDF is provided as an additional vector version.
- Retain the original captions or use `figure_captions_for_manuscript.txt`.
- The figures contain **test-only** descriptive means and should not be presented as direct proofs of statistical significance. Table 4 contains paired confidence intervals and inferential tests.
- **Do not** mix in Notebook 15/15b DC-Adapted data: those correspond to a separate external baseline comparison table, not the four-variant figures above.
